## EN LENGUAJE PYSPARK

In [71]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, LongType, FloatType, DoubleType, DecimalType, StringType, StructType, StructField
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.stat import Correlation
from pyspark.sql import Row

In [39]:
# 1. Inicializar la Sesión de Spark
# Asegúrate de tener PySpark instalado y configurado.
spark = SparkSession.builder \
    .appName("Analisis_Clientes_Behavioural") \
    .getOrCreate()


behavioural_df = spark.read.parquet("/home/jovyan/work/data/BEHAVIOURAL_CLEAN", header=True, inferSchema=True)
clientes_df = spark.read.parquet("/home/jovyan/work/data/CLIENTS_CLEAN", header=True, inferSchema=True)

In [40]:
## BEHAVIOURAL

# Reemplaza behavioural_psdf.head() con la función nativa show()
print("Primeras 5 filas de BEHAVIOURAL:")
behavioural_df.show(5, truncate=False)

# Además, añadimos printSchema() para revisar tipos (fundamental en Big Data)
print("Esquema de BEHAVIOURAL:")
behavioural_df.printSchema()

Primeras 5 filas de BEHAVIOURAL:
+------------------+------------+----------+--------------------+-----------------+------------------------+--------------------+------------------------+--------------------------+-------------------+-------------------+---------------+------------------+
|CONTRACT_ID       |CLIENT_ID   |DATE      |CREDICT_CARD_BALANCE|CREDIT_CARD_LIMIT|CREDIT_CARD_DRAWINGS_ATM|CREDIT_CARD_DRAWINGS|CREDIT_CARD_DRAWINGS_POS|CREDIT_CARD_DRAWINGS_OTHER|CREDIT_CARD_PAYMENT|NUMBER_DRAWINGS_ATM|NUMBER_DRAWINGS|NUMBER_INSTALMENTS|
+------------------+------------+----------+--------------------+-----------------+------------------------+--------------------+------------------------+--------------------------+-------------------+-------------------+---------------+------------------+
|ES1821016961u00XXX|ES182147947X|2020-08-22|0.0                 |2700.0           |0.0                     |0.0                 |0.0                     |0.0                       |0.0            

In [41]:
## CLIENTS

# Reemplaza clients_psdf.head() con la función nativa show()
print("Primeras 5 filas de CLIENTS:")
clientes_df.show(5, truncate=False)

# Además, añadimos printSchema() para revisar tipos (fundamental en Big Data)
print("Esquema de CLIENTS:")
clientes_df.printSchema()

Primeras 5 filas de CLIENTS:
+------------+----------------------+-----------------+------+------------+--------------+-----------+---------+--------------+--------------+------------+------------------+-------------+--------------+-----------+-----------------+-----------+------------------+------------------+---------------------+------------------+----------+--------------+----------+--------------------------+---------------------+------------------------+------------------------+------------------------+---------------------------+---------------------------+---------------------------+-----------------------+-----------------------+-----------------------+----------------------+----------------------+-------------------+---------------------+-----------------+-------------------+----------------+
|CLIENT_ID   |NON_COMPLIANT_CONTRACT|NAME_PRODUCT_TYPE|GENDER|TOTAL_INCOME|AMOUNT_PRODUCT|INSTALLMENT|EDUCATION|MARITAL_STATUS|HOME_SITUATION|REGION_SCORE|AGE_IN_YEARS      |JOB_SENIORIT

In [42]:
# --------- BEHAVIOURAL ---------
# Columnas numéricas
numeric_types = (IntegerType, LongType, FloatType, DoubleType, DecimalType)

behavioural_num_df = behavioural_df.select(
    *[f.name for f in behavioural_df.schema.fields 
      if isinstance(f.dataType, numeric_types)]
)

# Columnas "categóricas" (en Spark normalmente son StringType)
behavioural_cat_df = behavioural_df.select(
    *[f.name for f in behavioural_df.schema.fields 
      if isinstance(f.dataType, StringType)]
)

# --------- CLIENTES ---------
clientes_num_df = clientes_df.select(
    *[f.name for f in clientes_df.schema.fields 
      if isinstance(f.dataType, numeric_types)]
)

clientes_cat_df = clientes_df.select(
    *[f.name for f in clientes_df.schema.fields 
      if isinstance(f.dataType, StringType)]
)

In [43]:
print("Numéricas BEHAVIOURAL:", behavioural_num_df.columns)
print("Categorías BEHAVIOURAL:", behavioural_cat_df.columns)
print("Numéricas CLIENTES:", clientes_num_df.columns)
print("Categorías CLIENTES:", clientes_cat_df.columns)

Numéricas BEHAVIOURAL: ['CREDICT_CARD_BALANCE', 'CREDIT_CARD_LIMIT', 'CREDIT_CARD_DRAWINGS_ATM', 'CREDIT_CARD_DRAWINGS', 'CREDIT_CARD_DRAWINGS_POS', 'CREDIT_CARD_DRAWINGS_OTHER', 'CREDIT_CARD_PAYMENT', 'NUMBER_DRAWINGS_ATM', 'NUMBER_DRAWINGS', 'NUMBER_INSTALMENTS']
Categorías BEHAVIOURAL: ['CONTRACT_ID', 'CLIENT_ID']
Numéricas CLIENTES: ['NON_COMPLIANT_CONTRACT', 'TOTAL_INCOME', 'AMOUNT_PRODUCT', 'INSTALLMENT', 'REGION_SCORE', 'AGE_IN_YEARS', 'JOB_SENIORITY', 'HOME_SENIORITY', 'LAST_UPDATE', 'FAMILY_SIZE', 'PROACTIVE_SCORING', 'BEHAVIORAL_SCORING', 'DAYS_LAST_INFO_CHANGE', 'NUMBER_OF_PRODUCTS', 'DIGITAL_CLIENT', 'NUM_PREVIOUS_LOAN_APP', 'LOAN_ANNUITY_PAYMENT_MAX', 'LOAN_ANNUITY_PAYMENT_MIN', 'LOAN_ANNUITY_PAYMENT_SUM', 'LOAN_APPLICATION_AMOUNT_MAX', 'LOAN_APPLICATION_AMOUNT_MIN', 'LOAN_APPLICATION_AMOUNT_SUM', 'LOAN_CREDIT_GRANTED_MAX', 'LOAN_CREDIT_GRANTED_MIN', 'LOAN_CREDIT_GRANTED_SUM', 'LOAN_VARIABLE_RATE_MAX', 'LOAN_VARIABLE_RATE_MIN', 'NUM_STATUS_ANNULLED', 'NUM_STATUS_AUTHORIZED

In [44]:
def calcular_estadisticos_spark(clientes_df):
    
    filas_resumen = []

    for c in clientes_df.columns:

        # MEDIA
        media = clientes_df.select(F.mean(c)).first()[0]

        # MEDIANA
        mediana = clientes_df.approxQuantile(c, [0.5], 0.01)[0]

        # MODA
        moda_row = (
            clientes_df.groupBy(c)
                       .count()
                       .orderBy(F.desc("count"))
                       .first()
        )
        moda = moda_row[0] if moda_row else None

        # MIN & MAX
        stats_minmax = clientes_df.select(F.min(c), F.max(c)).first()
        minimo, maximo = stats_minmax[0], stats_minmax[1]
        rango = maximo - minimo if (minimo is not None and maximo is not None) else None

        # VARIANZA y STD
        stats_var_std = clientes_df.select(F.variance(c), F.stddev(c)).first()
        varianza, desv_std = stats_var_std[0], stats_var_std[1]

        # COEFICIENTE DE VARIACIÓN
        coef_var = desv_std / media if (media not in (None, 0)) else None

        # 👇 Cast a float (double) todo lo numérico
        filas_resumen.append({
            "Columna": str(c),
            "Media": float(media) if media is not None else None,
            "Mediana": float(mediana) if mediana is not None else None,
            "Moda": float(moda) if moda is not None else None,
            "Min": float(minimo) if minimo is not None else None,
            "Max": float(maximo) if maximo is not None else None,
            "Rango": float(rango) if rango is not None else None,
            "Varianza": float(varianza) if varianza is not None else None,
            "DesvStd": float(desv_std) if desv_std is not None else None,
            "CoefVar": float(coef_var) if coef_var is not None else None
        })

    # 📌 Definimos el esquema explícito
    schema = StructType([
        StructField("Columna",  StringType(),  True),
        StructField("Media",    DoubleType(),  True),
        StructField("Mediana",  DoubleType(),  True),
        StructField("Moda",     DoubleType(),  True),
        StructField("Min",      DoubleType(),  True),
        StructField("Max",      DoubleType(),  True),
        StructField("Rango",    DoubleType(),  True),
        StructField("Varianza", DoubleType(),  True),
        StructField("DesvStd",  DoubleType(),  True),
        StructField("CoefVar",  DoubleType(),  True),
    ])

    resumen_clientes = spark.createDataFrame(filas_resumen, schema=schema)

    return resumen_clientes


In [45]:
resumen_clientes = calcular_estadisticos_spark(clientes_num_df)
resumen_clientes.show(truncate=False)

+---------------------------+-------------------+------------------+------------------+---------------------+------------------+------------------+---------------------+--------------------+-------------------+
|Columna                    |Media              |Mediana           |Moda              |Min                  |Max               |Rango             |Varianza             |DesvStd             |CoefVar            |
+---------------------------+-------------------+------------------+------------------+---------------------+------------------+------------------+---------------------+--------------------+-------------------+
|NON_COMPLIANT_CONTRACT     |0.08244113431945373|0.0               |0.0               |0.0                  |1.0               |1.0               |0.0756450842340556   |0.27503651436501225 |3.3361563573260002 |
|TOTAL_INCOME               |2011.3439081883423 |1728.0            |1620.0            |307.8                |1404000.0         |1403692.2         |1.4390254

In [46]:
resumen_clientes.printSchema()

root
 |-- Columna: string (nullable = true)
 |-- Media: double (nullable = true)
 |-- Mediana: double (nullable = true)
 |-- Moda: double (nullable = true)
 |-- Min: double (nullable = true)
 |-- Max: double (nullable = true)
 |-- Rango: double (nullable = true)
 |-- Varianza: double (nullable = true)
 |-- DesvStd: double (nullable = true)
 |-- CoefVar: double (nullable = true)



In [47]:
# 1️⃣ Filtrar columnas categóricas con ≤ 20 categorías
columnas_validas = []
for c in clientes_cat_df.columns:
    n_cat = clientes_cat_df.select(F.countDistinct(c)).first()[0]
    if n_cat <= 20:
        columnas_validas.append(c)

print("Columnas categóricas con ≤ 20 categorías:")
print(columnas_validas)


Columnas categóricas con ≤ 20 categorías:
['NAME_PRODUCT_TYPE', 'GENDER', 'EDUCATION', 'MARITAL_STATUS', 'HOME_SITUATION', 'OWN_INSURANCE_CAR', 'OCCUPATION', 'HOME_OWNER']


In [48]:
for c in columnas_validas:
    print(f"\n===== Frecuencia de {c} =====")
    frecuencias = (
        clientes_cat_df
        .groupBy(c)
        .count()
        .orderBy(F.desc("count"))
    )
    frecuencias.show(truncate=False)


===== Frecuencia de NAME_PRODUCT_TYPE =====
+-----------------+------+
|NAME_PRODUCT_TYPE|count |
+-----------------+------+
|PRODUCT 1        |140256|
|PRODUCT 2        |13951 |
+-----------------+------+


===== Frecuencia de GENDER =====
+------+------+
|GENDER|count |
+------+------+
|F     |102168|
|M     |52039 |
+------+------+


===== Frecuencia de EDUCATION =====
+---------------------+------+
|EDUCATION            |count |
+---------------------+------+
|Secondary            |110994|
|NULL                 |36058 |
|Incomplete University|5118  |
|Primary School       |1953  |
|Master/PhD           |84    |
+---------------------+------+


===== Frecuencia de MARITAL_STATUS =====
+--------------+------+
|MARITAL_STATUS|count |
+--------------+------+
|Married       |113882|
|Single        |40325 |
+--------------+------+


===== Frecuencia de HOME_SITUATION =====
+-----------------------+------+
|HOME_SITUATION         |count |
+-----------------------+------+
|House          

In [49]:
for col in clientes_num_df.columns:
    print(f"\n===== Histograma de {col} =====")
    
    # Obtener una lista de valores de la columna como RDD
    rdd = clientes_num_df.select(col).rdd.flatMap(lambda x: x)
    
    # Histograma en Spark: 30 bins
    bins, counts = rdd.histogram(30)

    # Mostrar resultado en forma tabular
    hist_df = spark.createDataFrame([
        (float(bins[i]), float(bins[i+1]), int(counts[i]))
        for i in range(len(counts))
    ], ["bin_start", "bin_end", "count"])

    hist_df.show(truncate=False)



===== Histograma de NON_COMPLIANT_CONTRACT =====
+-------------------+-------------------+------+
|bin_start          |bin_end            |count |
+-------------------+-------------------+------+
|0.0                |0.03333333333333333|141494|
|0.03333333333333333|0.06666666666666667|0     |
|0.06666666666666667|0.1                |0     |
|0.1                |0.13333333333333333|0     |
|0.13333333333333333|0.16666666666666666|0     |
|0.16666666666666666|0.2                |0     |
|0.2                |0.23333333333333334|0     |
|0.23333333333333334|0.26666666666666666|0     |
|0.26666666666666666|0.3                |0     |
|0.3                |0.3333333333333333 |0     |
|0.3333333333333333 |0.36666666666666664|0     |
|0.36666666666666664|0.4                |0     |
|0.4                |0.43333333333333335|0     |
|0.43333333333333335|0.4666666666666667 |0     |
|0.4666666666666667 |0.5                |0     |
|0.5                |0.5333333333333333 |0     |
|0.5333333333333333

### CORRELACIÓN CLIENTS CON BEHAVIOURAL

In [ ]:
# --------- FUNCIÓN GENERAL PARA MATRIZ DE CORRELACIÓN ---------
def compute_corr_matrix(df, df_name="df"):
    """
    Calcula y muestra por pantalla la matriz de correlación de Pearson
    de un DataFrame con solo columnas numéricas, eliminando filas con nulls.
    """
    numeric_cols = df.columns
    if len(numeric_cols) == 0:
        print(f"No hay columnas numéricas en {df_name}")
        return

    # 1) Eliminamos filas que tengan null en alguna de las columnas numéricas
    df_no_nulls = df.na.drop(subset=numeric_cols)

    # 2) Montamos el vector de características
    assembler = VectorAssembler(
        inputCols=numeric_cols,
        outputCol="features"  # handleInvalid por defecto es "error", pero ya no hay nulls
    )
    vector_df = assembler.transform(df_no_nulls).select("features")

    # 3) Correlación de Pearson
    corr_matrix = Correlation.corr(vector_df, "features", "pearson").head()[0]
    corr_array = corr_matrix.toArray().tolist()  # lista de listas de Python

    # 🔹 NUEVA VISUALIZACIÓN EN TABLA 🔹
    corr_df = pd.DataFrame(corr_array, columns=numeric_cols, index=numeric_cols)

    print(f"\nMatriz de correlación para {df_name}:")
    display(corr_df)   # en Jupyter/Notebook se ve como tabla

    return numeric_cols, corr_array

# --------- CÁLCULO PARA CADA DATASET ---------
beh_cols, beh_corr = compute_corr_matrix(behavioural_num_df, "behavioural_num_df")
cli_cols, cli_corr = compute_corr_matrix(clientes_num_df, "clientes_num_df")



Matriz de correlación para behavioural_num_df:


,CREDICT_CARD_BALANCE,CREDIT_CARD_LIMIT,CREDIT_CARD_DRAWINGS_ATM,CREDIT_CARD_DRAWINGS,CREDIT_CARD_DRAWINGS_POS,CREDIT_CARD_DRAWINGS_OTHER,CREDIT_CARD_PAYMENT,NUMBER_DRAWINGS_ATM,NUMBER_DRAWINGS,NUMBER_INSTALMENTS
CREDICT_CARD_BALANCE,1.000000,0.502851,0.297704,0.338085,0.180017,0.068839,0.170358,0.330442,0.259991,0.035992
CREDIT_CARD_LIMIT,0.502851,1.000000,0.204303,0.270701,0.198381,0.042330,0.243413,0.181497,0.212744,-0.127986
CREDIT_CARD_DRAWINGS_ATM,0.297704,0.204303,1.000000,0.813612,0.088916,0.019076,0.179762,0.735354,0.308227,-0.071134
CREDIT_CARD_DRAWINGS,0.338085,0.270701,0.813612,1.000000,0.602822,0.241482,0.303970,0.607439,0.526878,-0.094927
CREDIT_CARD_DRAWINGS_POS,0.180017,0.198381,0.088916,0.602822,1.000000,0.009370,0.294566,0.083150,0.538200,-0.067656
CREDIT_CARD_DRAWINGS_OTHER,0.068839,0.042330,0.019076,0.241482,0.009370,1.000000,0.024187,0.012822,0.023441,-0.020800
CREDIT_CARD_PAYMENT,0.170358,0.243413,0.179762,0.303970,0.294566,0.024187,1.000000,0.145319,0.224442,-0.006516
NUMBER_DRAWINGS_ATM,0.330442,0.181497,0.735354,0.607439,0.083150,0.012822,0.145319,1.000000,0.422501,-0.054099
NUMBER_DRAWINGS,0.259991,0.212744,0.308227,0.526878,0.538200,0.023441,0.224442,0.422501,1.000000,-0.087439
NUMBER_INSTALMENTS,0.035992,-0.127986,-0.071134,-0.094927,-0.067656,-0.020800,-0.006516,-0.054099,-0.087439,1.000000



Matriz de correlación para clientes_num_df:


,NON_COMPLIANT_CONTRACT,TOTAL_INCOME,AMOUNT_PRODUCT,INSTALLMENT,REGION_SCORE,AGE_IN_YEARS,JOB_SENIORITY,HOME_SENIORITY,LAST_UPDATE,FAMILY_SIZE,...,LOAN_CREDIT_GRANTED_MAX,LOAN_CREDIT_GRANTED_MIN,LOAN_CREDIT_GRANTED_SUM,LOAN_VARIABLE_RATE_MAX,LOAN_VARIABLE_RATE_MIN,NUM_STATUS_ANNULLED,NUM_STATUS_AUTHORIZED,NUM_STATUS_DENIED,NUM_STATUS_NOT_USED,NUM_FLAG_INSURED
NON_COMPLIANT_CONTRACT,1.000000,0.003119,-0.036750,-0.016640,-0.037102,-0.069615,-0.076614,-0.037463,-0.037169,-0.000404,...,-0.007013,-0.020498,0.013332,-0.043887,-0.016855,0.028390,-0.027459,0.072552,0.002041,-0.005881
TOTAL_INCOME,0.003119,1.000000,0.089703,0.109630,0.038763,0.011411,0.004082,-0.007004,0.005468,-0.000320,...,0.063179,0.028447,0.049091,0.009342,-0.002744,0.013099,0.008081,0.011881,0.007432,0.006596
AMOUNT_PRODUCT,-0.036750,0.089703,1.000000,0.762599,0.087060,0.153382,0.087154,0.022986,0.030180,0.036236,...,0.163071,0.097687,0.102595,0.035458,0.015466,-0.021507,0.021659,-0.046734,-0.012573,0.003256
INSTALLMENT,-0.016640,0.109630,0.762599,1.000000,0.105442,0.085241,0.045414,-0.007602,0.021810,0.042932,...,0.158806,0.098710,0.111654,0.034136,0.011900,0.003969,0.009995,-0.008076,-0.013307,0.006313
REGION_SCORE,-0.037102,0.038763,0.087060,0.105442,1.000000,0.047748,-0.002128,0.057375,0.006444,-0.025979,...,0.067852,0.053984,0.051598,-0.000996,0.003297,0.002272,0.007508,0.004009,0.021901,0.028595
AGE_IN_YEARS,-0.069615,0.011411,0.153382,0.085241,0.047748,1.000000,0.348962,0.302950,0.078567,-0.204143,...,0.182765,0.036111,0.151246,-0.018931,-0.013308,0.066288,0.062231,0.006949,-0.066935,0.170497
JOB_SENIORITY,-0.076614,0.004082,0.087154,0.045414,-0.002128,0.348962,1.000000,0.174127,0.070579,-0.040435,...,0.076961,0.019583,0.059909,0.027109,0.011267,-0.000957,0.058235,-0.021095,-0.019947,0.087632
HOME_SENIORITY,-0.037463,-0.007004,0.022986,-0.007602,0.057375,0.302950,0.174127,1.000000,0.023107,-0.158784,...,0.030894,-0.000443,0.022627,-0.003828,0.009663,0.007490,0.010380,-0.027971,-0.019555,0.063730
LAST_UPDATE,-0.037169,0.005468,0.030180,0.021810,0.006444,0.078567,0.070579,0.023107,1.000000,0.104470,...,0.020895,-0.003722,0.015844,0.029007,0.002752,-0.000559,0.047186,-0.021202,0.001144,0.049710
FAMILY_SIZE,-0.000404,-0.000320,0.036236,0.042932,-0.025979,-0.204143,-0.040435,-0.158784,0.104470,1.000000,...,-0.004022,-0.014511,-0.018808,0.016362,-0.005872,-0.019469,0.025542,-0.019908,0.007439,-0.019615


In [ ]:
def get_high_corr_pairs_abs(cols, corr_array, threshold=0.4):
    """
    Devuelve pares de variables con |correlación| >= threshold.
    cols: lista de nombres de columnas
    corr_array: lista de listas (matriz de correlación)
    threshold: umbral (por defecto 0.4)
    """
    pairs = []
    n = len(cols)

    for i in range(n):
        for j in range(i + 1, n):  # solo triángulo superior, sin duplicar
            val = corr_array[i][j]
            if abs(val) >= threshold:
                pairs.append((cols[i], cols[j], float(val), float(abs(val))))

    return pairs


# -------- BEHAVIOURAL --------
beh_pairs = get_high_corr_pairs_abs(beh_cols, beh_corr, threshold=0.4)

beh_pairs_df = spark.createDataFrame(
    beh_pairs,
    ["Variable_1", "Variable_2", "Correlacion", "Abs_Correlacion"]
).orderBy(F.desc("Abs_Correlacion"))

print("📌 Correlaciones |r| >= 0.4 en BEHAVIOURAL:")
beh_pairs_df.show(beh_pairs_df.count(), truncate=False)   # 👈 aquí el cambio


# -------- CLIENTS --------
cli_pairs = get_high_corr_pairs_abs(cli_cols, cli_corr, threshold=0.4)

cli_pairs_df = spark.createDataFrame(
    cli_pairs,
    ["Variable_1", "Variable_2", "Correlacion", "Abs_Correlacion"]
).orderBy(F.desc("Abs_Correlacion"))

print("📌 Correlaciones |r| >= 0.4 en CLIENTS:")
cli_pairs_df.show(cli_pairs_df.count(), truncate=False)   # 👈 aquí el cambio


📌 Correlaciones |r| >= 0.4 en BEHAVIOURAL:
+------------------------+------------------------+-------------------+-------------------+
|Variable_1              |Variable_2              |Correlacion        |Abs_Correlacion    |
+------------------------+------------------------+-------------------+-------------------+
|CREDIT_CARD_DRAWINGS_ATM|CREDIT_CARD_DRAWINGS    |0.8136121525765816 |0.8136121525765816 |
|CREDIT_CARD_DRAWINGS_ATM|NUMBER_DRAWINGS_ATM     |0.7353535872130808 |0.7353535872130808 |
|CREDIT_CARD_DRAWINGS    |NUMBER_DRAWINGS_ATM     |0.6074391192003389 |0.6074391192003389 |
|CREDIT_CARD_DRAWINGS    |CREDIT_CARD_DRAWINGS_POS|0.6028223817623051 |0.6028223817623051 |
|CREDIT_CARD_DRAWINGS_POS|NUMBER_DRAWINGS         |0.5382002512739494 |0.5382002512739494 |
|CREDIT_CARD_DRAWINGS    |NUMBER_DRAWINGS         |0.5268784824853892 |0.5268784824853892 |
|CREDICT_CARD_BALANCE    |CREDIT_CARD_LIMIT       |0.5028514274046885 |0.5028514274046885 |
|NUMBER_DRAWINGS_ATM     |NUMBER_DRAW

UNIÓN DE LOS DATASETS

In [67]:
# Unión de datasets (inner, puedes usar left o full si quieres)
combined_df = behavioural_df.join(clientes_df, on="CLIENT_ID", how="inner")

In [68]:
combined_num_df = combined_df.select(
    *[f.name for f in combined_df.schema.fields if isinstance(f.dataType, numeric_types)]
)

In [ ]:
# Ya tienes esto:
combined_cols, combined_corr = compute_corr_matrix(
    combined_num_df,
    "combined_df"
)

# ---- Visualización en forma de tabla con Spark ----
rows = []
for row_vals, row_name in zip(combined_corr, combined_cols):
    fila = {"FEATURE": row_name}
    fila.update({col: float(val) for col, val in zip(combined_cols, row_vals)})
    rows.append(Row(**fila))

corr_spark_df = spark.createDataFrame(rows)


Matriz de correlación para combined_df:


,CREDICT_CARD_BALANCE,CREDIT_CARD_LIMIT,CREDIT_CARD_DRAWINGS_ATM,CREDIT_CARD_DRAWINGS,CREDIT_CARD_DRAWINGS_POS,CREDIT_CARD_DRAWINGS_OTHER,CREDIT_CARD_PAYMENT,NUMBER_DRAWINGS_ATM,NUMBER_DRAWINGS,NUMBER_INSTALMENTS,...,LOAN_CREDIT_GRANTED_MAX,LOAN_CREDIT_GRANTED_MIN,LOAN_CREDIT_GRANTED_SUM,LOAN_VARIABLE_RATE_MAX,LOAN_VARIABLE_RATE_MIN,NUM_STATUS_ANNULLED,NUM_STATUS_AUTHORIZED,NUM_STATUS_DENIED,NUM_STATUS_NOT_USED,NUM_FLAG_INSURED
CREDICT_CARD_BALANCE,1.000000,0.506667,0.301475,0.344166,0.187952,0.069232,0.166748,0.332938,0.266631,0.032488,...,0.094712,0.009671,0.088667,-0.030789,-0.042966,0.043284,0.010219,0.049778,-0.018644,0.007830
CREDIT_CARD_LIMIT,0.506667,1.000000,0.209244,0.276215,0.201711,0.042515,0.245514,0.186858,0.218232,-0.132341,...,0.166921,0.005861,0.133115,0.008545,-0.062676,0.059662,0.070585,0.016925,-0.018967,0.003223
CREDIT_CARD_DRAWINGS_ATM,0.301475,0.209244,1.000000,0.809980,0.093782,0.020957,0.174191,0.739231,0.310671,-0.071636,...,0.050441,-0.001125,0.047022,-0.008009,-0.018351,0.032823,0.011596,0.020074,-0.008501,0.001812
CREDIT_CARD_DRAWINGS,0.344166,0.276215,0.809980,1.000000,0.613482,0.241794,0.295390,0.608158,0.529055,-0.094521,...,0.061534,-0.001418,0.053479,-0.006378,-0.022373,0.047350,0.009286,0.021482,-0.007520,-0.018668
CREDIT_CARD_DRAWINGS_POS,0.187952,0.201711,0.093782,0.613482,1.000000,0.010533,0.285346,0.087410,0.535311,-0.066808,...,0.038135,-0.000634,0.028338,-0.000779,-0.014882,0.039429,0.000142,0.008782,-0.001005,-0.036235
CREDIT_CARD_DRAWINGS_OTHER,0.069232,0.042515,0.020957,0.241794,0.010533,1.000000,0.025750,0.013983,0.024315,-0.019831,...,0.011812,-0.000534,0.011231,0.000295,-0.002040,0.005030,0.001242,0.007157,-0.002335,-0.002513
CREDIT_CARD_PAYMENT,0.166748,0.245514,0.174191,0.295390,0.285346,0.025750,1.000000,0.144274,0.219405,-0.007395,...,0.065932,0.008028,0.057460,-0.006088,-0.019438,0.035353,0.011627,0.020677,-0.007641,-0.006352
NUMBER_DRAWINGS_ATM,0.332938,0.186858,0.739231,0.608158,0.087410,0.013983,0.144274,1.000000,0.421836,-0.057290,...,0.030585,-0.007826,0.031907,-0.016419,-0.022564,0.030589,0.010918,0.023467,-0.009291,0.009843
NUMBER_DRAWINGS,0.266631,0.218232,0.310671,0.529055,0.535311,0.024315,0.219405,0.421836,1.000000,-0.086140,...,0.017435,-0.016226,0.017651,-0.015449,-0.023025,0.059560,0.001954,0.021028,-0.001183,-0.033321
NUMBER_INSTALMENTS,0.032488,-0.132341,-0.071636,-0.094521,-0.066808,-0.019831,-0.007395,-0.057290,-0.086140,1.000000,...,0.019411,0.073986,0.002752,-0.072276,0.008980,-0.036979,-0.078184,0.003673,-0.016338,0.044848


In [66]:
# Función para pares con |correlación| >= threshold
def get_high_corr_pairs_abs(cols, corr_array, threshold=0.4):
    """
    Devuelve pares de variables con |correlación| >= threshold.
    cols: lista de nombres de columnas
    corr_array: lista de listas (matriz de correlación)
    threshold: umbral (por defecto 0.4)
    """
    pairs = []
    n = len(cols)

    for i in range(n):
        for j in range(i + 1, n):  # solo triángulo superior, sin duplicar
            val = corr_array[i][j]
            if abs(val) >= threshold:
                pairs.append((cols[i], cols[j], float(val), float(abs(val))))

    return pairs

# 👉 Ya tienes esto calculado antes:
# combined_cols, combined_corr = compute_corr_matrix(combined_num_df, "combined_df")

# Obtener pares con |r| >= 0.4
comb_pairs = get_high_corr_pairs_abs(combined_cols, combined_corr, threshold=0.4)

# Pasar a Spark DataFrame
comb_pairs_df = spark.createDataFrame(
    comb_pairs,
    ["Variable_1", "Variable_2", "Correlacion", "Abs_Correlacion"]
)

# Ordenar de mayor a menor correlación absoluta
comb_pairs_df = comb_pairs_df.orderBy(F.desc("Abs_Correlacion"))

print("📌 Correlaciones |r| >= 0.4 en COMBINED_DF:")
comb_pairs_df.show(comb_pairs_df.count(), truncate=False)  # muestra TODAS


📌 Correlaciones |r| >= 0.4 en COMBINED_DF:
+---------------------------+---------------------------+-------------------+-------------------+
|Variable_1                 |Variable_2                 |Correlacion        |Abs_Correlacion    |
+---------------------------+---------------------------+-------------------+-------------------+
|LOAN_APPLICATION_AMOUNT_SUM|LOAN_CREDIT_GRANTED_SUM    |0.9929952589967873 |0.9929952589967873 |
|LOAN_APPLICATION_AMOUNT_MAX|LOAN_CREDIT_GRANTED_MAX    |0.9837475590710765 |0.9837475590710765 |
|LOAN_ANNUITY_PAYMENT_SUM   |LOAN_CREDIT_GRANTED_SUM    |0.9112168917367232 |0.9112168917367232 |
|LOAN_ANNUITY_PAYMENT_SUM   |LOAN_APPLICATION_AMOUNT_SUM|0.9046727466235348 |0.9046727466235348 |
|LOAN_APPLICATION_AMOUNT_MIN|LOAN_CREDIT_GRANTED_MIN    |0.8802054637598613 |0.8802054637598613 |
|LOAN_ANNUITY_PAYMENT_MIN   |LOAN_CREDIT_GRANTED_MIN    |0.8400788833118353 |0.8400788833118353 |
|CREDIT_CARD_DRAWINGS_ATM   |CREDIT_CARD_DRAWINGS       |0.8099796864573929

Al pasarlo a PySpark hay un error del orden de 0.00x a 0.02–0.03 en muchos coeficientes.
Ejemplo que tú misma has puesto:

Spark: 0.5251

Python: 0.502851
→ diferencia ≈ 0.022

Eso no es enorme en términos de correlación:

- No cambia el signo (positiva sigue siendo positiva, negativa sigue siendo negativa).

- La interpretación “fuerte / débil / casi 0” suele ser la misma.

- Pero si estás comparando variables muy similares entre sí (por ejemplo para ordenar features por importancia) esas pequeñas diferencias sí podrían cambiar el ranking entre dos variables muy parecidas.